In [1]:
!mamba install pandas
%pip install joblib
!mamba install scikit-learn
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.925 seconds
  Name    Version  Build                Channel
-------------------------------------------------------------
+ pandas  3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
- pip     26.2.2   pyh145f28c_0         conda-forge
mambajs 0.21.4

Process pip requirements ...

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 2.460 seconds
  Name                Version         Build                    Channel
--------------------------------------------------------------------------------------------
+ brotli-python       1.2.0           py313h8faef1e_3          emscripten-forge-4x
+ ca-certificates     2026.10.23      hbd8a1cb_0               cond

In [2]:
df = pd.read_csv("../Data/Telco-Customer-Churn.csv")

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]

print(X.shape)
print(y.value_counts())

(7043, 19)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [3]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

/tmp/xpython_42/1724188592.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, random_state=42
    ),
    "Linear SVC": SVC(
        kernel="linear", probability=True, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        random_state=42
    )
}

In [6]:
results = []
trained_models = {}

for name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results)

print(results_df)

/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


           Model  Accuracy  Precision    Recall   ROC-AUC
0  Decision Tree  0.791341   0.634228  0.505348  0.835093
1     Linear SVC  0.787793   0.617555  0.526738  0.825232
2  Random Forest  0.777857   0.603390  0.475936  0.816159


In [7]:
results_df = results_df.sort_values(
    by=["Recall", "ROC-AUC"],
    ascending=False
)

champion_name = results_df.iloc[0]["Model"]
champion_model = trained_models[champion_name]

joblib.dump(
    champion_model,
    "../models/champion_churn_pipeline.joblib"
)

print("Champion Model:", champion_name)
print("Champion model saved successfully!")

Champion Model: Linear SVC
Champion model saved successfully!


In [8]:
import os

print(os.listdir("../models"))

['champion_churn_pipeline.joblib', 'churn_pipeline.joblib']
